# Natural Language Processing — Sentiment Analysis

## The Core Challenge: Computers Cannot Read

Machine learning models work with numbers. Text is not numbers. Before any ML algorithm can classify a restaurant review, we must convert raw text into a numerical representation that captures something meaningful about its content.

This notebook covers the **classical NLP pipeline** — the approach that dominated before deep learning transformers:

```
Raw text
  → Clean and normalise
  → Convert to numbers (Bag of Words)
  → Train a classifier
  → Predict sentiment
```

---

## The Task: Restaurant Review Sentiment

Given a restaurant review, predict whether it is **positive (1)** or **negative (0)**.

Examples from the dataset:

| Review | Sentiment |
|--------|----------|
| "Wow... Loved this place." | 1 (Positive) |
| "Crust is not good." | 0 (Negative) |
| "Not tasty and the texture was just nasty." | 0 (Negative) |
| "Stopped by during the late May bank holiday off Rick Steve recommendation." | 1 (Positive) |

This is a **binary text classification** problem.

---

## The Pipeline We Will Build

1. **Text cleaning** — remove punctuation, lowercase, remove common words, stem words to their root
2. **Bag of Words** — convert each review to a vector of word counts
3. **Train Naive Bayes** — a classifier that works well with word count features
4. **Evaluate** — confusion matrix and accuracy on unseen reviews

## Step 1: Import Libraries

| Library | Why we need it |
|---------|---------------|
| `numpy` | Array operations |
| `matplotlib` | Plotting (optional here) |
| `pandas` | Loading the TSV dataset |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Step 2: Load the Dataset

The dataset is a **TSV (Tab-Separated Values)** file — we use `delimiter='\t'` instead of the default comma.

`quoting=3` disables quote handling (`csv.QUOTE_NONE`). Restaurant reviews often contain quote characters like `"` that would confuse the CSV parser if we left quoting enabled.

The dataset contains **1,000 reviews** with two columns:
- `Review`: the raw text
- `Liked`: 1 (positive) or 0 (negative)

The classes are perfectly balanced: 500 positive, 500 negative. This is ideal for a learning exercise — in real-world sentiment datasets, negative reviews are often rarer.

In [ ]:
dataset = pd.read_csv('Restaurant_Reviews.tsv', delimiter = '\t', quoting = 3)

## Step 3: Clean the Text

Raw text contains a lot of noise that would hurt our model. We apply a standard NLP cleaning pipeline to each review:

### Why each step matters

**1. Remove non-alphabetic characters** (`re.sub('[^a-zA-Z]', ' ', review)`)

Punctuation, numbers, and special characters carry almost no sentiment signal for this task. `"Wow!!!"` and `"Wow"` mean the same thing. Removing them also prevents the word `"good"` and `"good."` from being treated as different words.

**2. Lowercase** (`review.lower()`)

`"Great"`, `"great"`, and `"GREAT"` all mean the same thing. Without lowercasing, the model treats them as three different words and cannot generalise between them.

**3. Remove stopwords** (from `nltk`)

Stopwords are extremely common words that appear in almost every sentence regardless of sentiment: `"the"`, `"a"`, `"is"`, `"was"`, `"I"`, `"it"`. They would dominate the word count vectors without carrying useful signal.

**Critical exception: we keep `"not"`**

`"not good"` is the opposite of `"good"`. Removing `"not"` would make those reviews indistinguishable. We manually remove `"not"` from the stopwords list.

**4. Stemming** (`PorterStemmer`)

Stemming reduces words to their root form:
- `"loved"`, `"loves"`, `"loving"` → `"love"`
- `"tasty"`, `"tastier"` → `"tasti"`
- `"running"` → `"run"`

Without stemming, these variations would be counted as different words, splitting the signal. Stemming is aggressive but fast — note that stems are not always real English words (`"tasti"`).

The result is a `corpus`: a list of 1,000 cleaned strings, one per review.

In [ ]:
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
corpus = []
for i in range(0, 1000):
  review = re.sub('[^a-zA-Z]', ' ', dataset['Review'][i])
  review = review.lower()
  review = review.split()
  ps = PorterStemmer()
  all_stopwords = stopwords.words('english')
  all_stopwords.remove('not')
  review = [ps.stem(word) for word in review if not word in set(all_stopwords)]
  review = ' '.join(review)
  corpus.append(review)

In [ ]:
print(corpus)

## Step 4: Create the Bag of Words Model

Now we convert the cleaned text into numbers that a machine learning model can process.

**What is Bag of Words?**

Bag of Words (BoW) creates a **vocabulary** of the most frequent words across all reviews, then represents each review as a vector of word counts.

Example with a tiny vocabulary:

```
Vocabulary: [food, great, bad, service, love]

Review: "great food great service"   →  [2, 1, 0, 1, 0]
Review: "bad food love here"         →  [1, 0, 1, 0, 1]
```

Each review becomes a row in a matrix where each column represents one word.

**Why `max_features=1500`?**

The full vocabulary from 1,000 reviews might have 3,000+ unique stems. Most rare words appear in only 1-2 reviews and add noise, not signal. We keep only the top 1,500 most frequent words — this creates an X matrix of shape **(1000 samples x 1500 features)**.

**What BoW loses:**

| Lost information | Example |
|-----------------|--------|
| Word order | "food not good" vs "good not food" look identical |
| Context | "not bad" = positive, but "not" and "bad" in isolation look negative |
| Semantics | "great" and "excellent" are different words despite same meaning |

Modern NLP (transformers, word embeddings) addresses these limitations. BoW is the historical baseline that is still surprisingly effective for simple sentiment tasks.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 1500)
X = cv.fit_transform(corpus).toarray()
y = dataset.iloc[:, -1].values

## Step 5: Train/Test Split

Standard 80/20 split: 800 reviews for training, 200 for testing.

**Note:** We split *after* building the Bag of Words. This means the vocabulary (`cv`) was fit on all 1,000 reviews — a subtle form of data leakage. The proper pipeline would be:

```python
# Correct order:
X_train_raw, X_test_raw = train_test_split(corpus, ...)
cv = CountVectorizer(max_features=1500)
X_train = cv.fit_transform(X_train_raw).toarray()  # fit on train only
X_test  = cv.transform(X_test_raw).toarray()        # transform test with train vocabulary
```

For a learning exercise with a balanced dataset, this leakage is minor. In production, always fit the vectoriser on the training set only.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

## Step 6: Train Naive Bayes

**Why Naive Bayes for text classification?**

Naive Bayes is a probabilistic classifier based on Bayes' theorem. It predicts the class with the highest probability given the observed word counts.

It is called "naive" because it assumes all features (words) are **independent** of each other, given the class. This is clearly wrong in language — words like `"not"` and `"good"` are not independent. Yet Naive Bayes works surprisingly well for text classification because:

1. Even if the probability estimates are wrong, the **ranking** of classes is often correct
2. The independence assumption means no feature interactions to overfit on
3. It trains in milliseconds on the 1,500-feature matrix

**GaussianNB** assumes features follow a Gaussian distribution. For text, `MultinomialNB` is actually more appropriate (since word counts are discrete), but `GaussianNB` still works here.

In [ ]:
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, y_train)

## Step 7: Make Predictions

Each row in the output is `[predicted, actual]`:
- `[1, 1]` = correctly predicted positive
- `[0, 0]` = correctly predicted negative
- `[1, 0]` = false positive (predicted positive, was actually negative)
- `[0, 1]` = false negative (predicted negative, was actually positive)

Scanning this output gives a qualitative sense of where the model makes mistakes.

In [ ]:
y_pred = classifier.predict(X_test)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

## Step 8: Evaluate — Confusion Matrix and Accuracy

The confusion matrix gives a complete picture of classification performance:

```
              Predicted Negative   Predicted Positive
Actual Negative      TN                  FP
Actual Positive      FN                  TP
```

**Reading the results (~73% accuracy):**

73% is a solid baseline for a classical text classification approach on noisy restaurant reviews. Humans rating sentiment from short reviews often disagree with each other ~15% of the time.

**Where does the model fail?**

- **False positives:** Reviews with positive words but negative intent (e.g., sarcasm: `"Oh wow, amazing service..."` said ironically)
- **False negatives:** Negative reviews that avoid common negative words
- **Word order blindness:** `"not bad"` = positive, but BoW treats `"not"` and `"bad"` as separate signals

**How to improve:**

| Technique | Expected gain |
|-----------|---------------|
| TF-IDF instead of raw counts | Downweights common words | +2-5% |
| Bigrams (word pairs) | Captures "not good" as one feature | +3-7% |
| Logistic Regression or SVM | Often outperforms NB on larger datasets | +3-8% |
| Pre-trained word embeddings | Captures semantic similarity | +10-15% |
| Fine-tuned BERT | State-of-the-art for sentiment | +15-25% |

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)